# RecoMart Model Training and Evaluation

This notebook invokes the production modeling pipeline. It preserves the prepared chronological split, trains collaborative Funk SVD and cosine content models, logs both experiments to MLflow, and displays the resulting metrics and recommendation examples.

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.modeling.config import load_modeling_config
from src.modeling.runner import TrainingRunner

config = load_modeling_config(root / 'configs/modeling.yaml')
print(f'Repository: {root}')
print(f'MLflow experiment: {config.experiment_name}')

## Train and evaluate

The runner resolves the latest successful feature batch, loads Feature Store tables, reuses the immutable train/validation/test files, and produces one versioned model run.

In [ ]:
summary = TrainingRunner(config).run(algorithm='all')
print('Model run:', summary['model_run_id'])
print('Feature batch:', summary['feature_batch_id'])
print('Status:', summary['status'])

In [ ]:
import pandas as pd

metrics = pd.DataFrame({
    name: result['metrics']
    for name, result in summary['models'].items()
}).T
metrics

## Recommendations

Scores have model-specific meaning: predicted explicit rating for collaborative filtering and cosine similarity for content-based retrieval.

In [ ]:
collaborative = summary['models']['collaborative']
print('Collaborative recommendations for user', collaborative['example_user_id'])
pd.DataFrame(collaborative['recommendation_example'])

In [ ]:
content = summary['models']['content_based']
print('Similar products for product', content['example_product_id'])
pd.DataFrame(content['recommendation_example'])

## Model comparison and MLflow lineage

In [ ]:
comparison = pd.DataFrame(
    summary['comparison']['comparison_table'][1:],
    columns=summary['comparison']['comparison_table'][0],
)
display(comparison)
print('Recommended model:', summary['comparison']['recommended_model'])
for name, result in summary['models'].items():
    print(name, 'MLflow run:', result['mlflow']['run_id'])
print('PDF:', next(path for path in summary['report_paths'] if path.endswith('.pdf')))